# Notebook 04c: Iterative CMF Representation Unlearning (`04c_cmf_adaptive.ipynb`)

### Objectives & Workflow:
1. **Start from 04b Post-Hoc Aligned Checkpoint**: Load the best checkpoint of Notebook 04b (`cmf_04b_{MODEL_NAME}_seed_{seed}.pt`) across models (`resnet18` / `vit11m`) and seeds.
2. **CMF Head Reconstruction & Tying**: Reconstruct classifier weights directly from centered & normalized class-mean features at each epoch and freeze the head $W$.
3. **Trainable Components**: Optimize only the encoder $\theta$ (encoder-only unlearning) with the CMF head frozen — same mechanism as 04a but starting from the 04b checkpoint.
4. **Validation-Driven Stopping**: Multi-stage continued training loop selecting checkpoints based on feature-level erasure (`1 - LP_f`, `1 - NCC_f`) under the retain accuracy constraint.

**Dependency**: Requires completed Notebook 01, 04a, and 04b artifacts.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    compute_output_accuracy,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    print_metrics_summary_table,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. CMF Head Reconstruction Utility

Identical to 04a: computes centered & normalized class-mean features from the encoder and assigns them as frozen classifier weights.

In [ ]:
@torch.no_grad()
def reconstruct_cmf_head(model, dataloader, num_classes=10, device="cuda"):
    """
    Computes class-mean features from the encoder, centers them against the global mean,
    normalizes, and assigns to the linear classifier head. Freezes head parameters.
    """
    feats, targets = extract_features(model.encoder, dataloader, device=device)
    class_means = []

    for c in range(num_classes):
        mask = (targets == c)
        if mask.sum() > 0:
            mean = feats[mask].mean(axis=0)
        else:
            mean = np.zeros(feats.shape[1])
        class_means.append(mean)

    class_means = np.stack(class_means, axis=0)  # [C, D]
    global_mean = class_means.mean(axis=0, keepdims=True)
    centered = class_means - global_mean
    norms = np.linalg.norm(centered, axis=1, keepdims=True) + 1e-8
    normalized_weights = centered / norms  # [C, D]

    weight_tensor = torch.tensor(normalized_weights, dtype=torch.float32, device=device)
    model.head.linear.weight.data.copy_(weight_tensor)
    if model.head.linear.bias is not None:
        model.head.linear.bias.data.zero_()

    # Freeze classifier head
    model.head.linear.weight.requires_grad = False
    if model.head.linear.bias is not None:
        model.head.linear.bias.requires_grad = False

## 2. Configuration & Dataset Setup

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # e.g. [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

# Multi-stage schedule: smaller LRs than 04a since we refine from the 04b checkpoint
STAGES = [
    {"lr": 1e-4, "max_epochs": 4, "patience": 2},
    {"lr": 2e-5, "max_epochs": 3, "patience": 2},
]
DELTA_RETAIN = 0.05  # Retain accuracy constraint δ
MIN_DELTA = 0.002

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/plots", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")

with open(split_file, "r") as f:
    split_info = json.load(f)

# Train loader: retain (true labels) + forget (random labels) — same as 04a
class RandomTargetDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, num_classes):
        self.base_dataset = base_dataset
        self.num_classes = num_classes
    def __len__(self):
        return len(self.base_dataset)
    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        img = item[0]
        rand_label = np.random.randint(0, self.num_classes)
        if len(item) == 3:
            return img, rand_label, item[2]
        return img, rand_label

forget_dataset = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)
forget_dataset_rl = RandomTargetDataset(forget_dataset, NUM_CLASSES)

retain_dataset = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
combined_dataset = ConcatDataset([forget_dataset_rl, retain_dataset])
train_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dataset: {DATASET_NAME} | Model: {MODEL_NAME} | Seeds: {MODEL_SEEDS}")
print(f"Retain train: {len(retain_dataset)} | Forget eval: {len(forget_eval_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

## 3. Validation-Driven Continued Training Engine for 04c

**Identical engine to 04a** — encoder-only training with CMF head rebuilt & frozen each epoch —
but initialized from the post-hoc classifier-aligned checkpoint of 04b (`cmf_04b_*.pt`).
Learning rates are smaller since the starting point already has suppressed output-level forget accuracy.

In [ ]:
all_04c_results = []

for seed in MODEL_SEEDS:
    # --- Load 04b checkpoint (required input) ---
    kaggle_ckpt_04b = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/cmf_04b_{MODEL_NAME}_seed_{seed}.pt"
    local_ckpt_04b = f"./artifacts/checkpoints/cmf_04b_{MODEL_NAME}_seed_{seed}.pt"
    ckpt_04b = kaggle_ckpt_04b if os.path.exists(kaggle_ckpt_04b) else local_ckpt_04b

    # --- Load original validation retain accuracy (for eligibility constraint) ---
    kaggle_metrics = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.json"
    local_metrics = f"./artifacts/metrics/original_{MODEL_NAME}_seed_{seed}.json"
    orig_metrics_file = kaggle_metrics if os.path.exists(kaggle_metrics) else local_metrics

    if not os.path.exists(ckpt_04b):
        raise FileNotFoundError(f"04b checkpoint missing at {kaggle_ckpt_04b} or {local_ckpt_04b}. Run Notebook 04b first.")
    if not os.path.exists(orig_metrics_file):
        raise FileNotFoundError(f"Original metrics missing at {kaggle_metrics} or {local_metrics}. Run Notebook 01 first.")
    print(f"Using 04b ckpt: {ckpt_04b} | orig metrics: {orig_metrics_file}")

    with open(orig_metrics_file, "r") as f:
        orig_val_retain = json.load(f)["validation_metrics"]["output_retain"]

    print(f"\n==================================================")
    print(f"--- Running 04c: Iterative CMF | Model: {MODEL_NAME} | Seed: {seed} ---")
    print(f"Original Val Retain (threshold): {orig_val_retain:.4f} | δ = {DELTA_RETAIN}")
    print(f"==================================================")

    # Load 04b model weights
    model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(ckpt_04b, map_location=device))

    criterion = nn.CrossEntropyLoss()
    best_snapshot = None
    start_time = time.time()

    # --- Multi-Stage Continued Training Loop (encoder-only, CMF head frozen) ---
    for stage_idx, stage in enumerate(STAGES):
        if best_snapshot is not None:
            model.load_state_dict(best_snapshot["state"])

        # Only encoder parameters are trainable; head is rebuilt & frozen each epoch
        optimizer = optim.SGD(model.encoder.parameters(), lr=stage["lr"], momentum=0.9, weight_decay=5e-4)
        no_gain = 0

        for epoch in range(1, stage["max_epochs"] + 1):
            # Rebuild CMF head from current encoder features using clean retain training set (D_R); freezes head in-place
            reconstruct_cmf_head(model, retain_eval_loader, num_classes=NUM_CLASSES, device=device)

            # Train encoder only; head stays in eval mode (frozen)
            model.encoder.train()
            model.head.eval()
            for images, targets in train_loader:
                images, targets = images.to(device), targets.to(device)
                optimizer.zero_grad()
                out = model(images)
                loss = criterion(out, targets)
                loss.backward()
                optimizer.step()

            # Evaluate on validation D_V — probe trained on clean D_R
            val_metrics = evaluate_three_tier_metrics(
                model=model,
                probe_train_loader=retain_eval_loader,
                retain_eval_loader=val_loader,
                forget_eval_loader=forget_eval_loader,
                num_classes=NUM_CLASSES,
                seed=seed,
                device=device
            )

            # Score = feature-level erasure (1 - LP_f, 1 - NCC_f) + retain accuracy
            score = 0.5 * (1.0 - val_metrics["lp_forget"]) + 0.5 * (1.0 - val_metrics["ncc_forget"]) + val_metrics["output_retain"]
            eligible = (val_metrics["output_retain"] >= orig_val_retain - DELTA_RETAIN)

            print(
                f"Stage {stage_idx+1} Ep {epoch} | "
                f"Val Retain: {val_metrics['output_retain']:.4f} | "
                f"LP Forget: {val_metrics['lp_forget']:.4f} | "
                f"NCC Forget: {val_metrics['ncc_forget']:.4f} | "
                f"Score: {score:.4f} (Eligible: {eligible})"
            )

            if eligible and (best_snapshot is None or score > best_snapshot["score"] + MIN_DELTA):
                best_snapshot = {
                    "score": score,
                    "metrics": val_metrics,
                    "state": copy.deepcopy(model.state_dict())
                }
                no_gain = 0
            else:
                no_gain += 1

            if no_gain >= stage["patience"]:
                print(f"Early stopping triggered in stage {stage_idx+1}")
                break

    # Restore final best checkpoint
    if best_snapshot is not None:
        model.load_state_dict(best_snapshot["state"])

    runtime = time.time() - start_time
    save_ckpt = f"./artifacts/checkpoints/cmf_04c_{MODEL_NAME}_seed_{seed}.pt"
    torch.save(model.state_dict(), save_ckpt)

    # Final evaluation on test set — exactly once
    test_metrics = evaluate_three_tier_metrics(
        model=model,
        probe_train_loader=retain_eval_loader,
        retain_eval_loader=retain_eval_loader,
        forget_eval_loader=forget_eval_loader,
        num_classes=NUM_CLASSES,
        seed=seed,
        device=device,
        test_eval_loader=test_loader
    )

    res = {
        "variant": "04c_cmf_iterative",
        "model_name": MODEL_NAME,
        "seed": seed,
        "split_seed": SPLIT_SEED,
        "runtime_sec": runtime,
        "validation_metrics": best_snapshot["metrics"] if best_snapshot else val_metrics,
        "test_metrics": test_metrics
    }
    all_04c_results.append(res)
    res_path = f"./artifacts/metrics/cmf_04c_{MODEL_NAME}_seed_{seed}.json"
    with open(res_path, "w") as f:
        json.dump(res, f, indent=2)

    print(f"04c Test Retain Output Acc: {test_metrics['output_retain']:.4f} | Forget: {test_metrics['output_forget']:.4f}")
    print(f"04c Test LP Forget Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
    print(f"04c Test NCC Forget Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
    print(f"Saved 04c checkpoint & metrics to {res_path}")

    # Feature space & decision boundary visualization for first seed only
    if seed == MODEL_SEEDS[0]:
        vis_save = f"./artifacts/plots/features_04c_cmf_{MODEL_NAME}_seed_{seed}.png"
        visualize_feature_space_and_boundaries(
            model=model,
            retain_loader=retain_eval_loader,
            forget_loader=forget_eval_loader,
            num_classes=NUM_CLASSES,
            forget_class=0,
            title=f"Feature Space & Linear Output: 04c CMF Iterative ({MODEL_NAME}) - Retain(D_R) vs Forget(D_F)",
            device=device,
            save_path=vis_save
        )

# ============================================================
# Final Consolidated Benchmark Metrics Summary Table
# ============================================================
summary_df = print_metrics_summary_table(
    results=all_04c_results,
    title="04c Iterative CMF Representation Unlearning Metrics Summary",
    save_csv_path=f"./artifacts/metrics/summary_04c_cmf_{MODEL_NAME}.csv"
)